In [10]:
"""
Synthetic Nigerian Traffic Congestion Dataset Generator
=========================================================

Reference schema (Kaggle: hasibullahaman/traffic-prediction-dataset):
    Time, Date, Day of the week, CarCount, BikeCount, BusCount, TruckCount,
    Total, Traffic Situation (1-Heavy, 2-High, 3-Normal, 4-Low)

This script extends that schema with Nigeria-specific segments, vehicle
categories (Keke/Okada instead of generic "Bike"), and exogenous drivers
(rainfall, market days, school hours, fuel scarcity) that are known
congestion drivers in Nigerian cities, calibrated loosely against
observed peak/off-peak ratios reported in local case studies
(Ibadan, Abuja, Port Harcourt, Ado-Ekiti junction studies).

Output: a CSV file with one row per (segment, 15-minute interval).
"""

import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# ----------------------------------------------------------------------
# 1. CONFIG
# ----------------------------------------------------------------------

SEED = 42
rng = np.random.default_rng(SEED)

START_DATE = datetime(2025, 1, 1)
NUM_DAYS = 60                # ~2 months of data
INTERVAL_MINUTES = 15
STATE = "Lagos"

# Road segments in Lagos, each with characteristics affecting baseline
# volume, capacity and congestion sensitivity.
SEGMENTS = [
    # name,                       road_type,   lanes, capacity(15min), centrality, market_axis, school_axis
    ("Third Mainland Bridge",     "highway",   6,     900,  0.95, False, False),
    ("Lekki-Epe Expressway",      "highway",   6,     800,  0.85, True,  True),
    ("Ikorodu Road",              "arterial",  4,     650,  0.90, True,  True),
    ("Agege Motor Road",          "arterial",  3,     500,  0.75, True,  True),
    ("Apapa-Oshodi Expressway",   "highway",   6,     850,  0.88, True,  False),
    ("Ozumba Mbadiwe",            "arterial",  4,     600,  0.70, False, False),
    ("Ajah-Sangotedo Road",       "arterial",  3,     450,  0.65, True,  True),
    ("Berger-Ojota Corridor",     "arterial",  4,     550,  0.72, True,  True),
]

# Hourly multipliers describing typical Lagos daily traffic rhythm
# (index = hour of day, 0-23). Weekday has sharp AM/PM peaks;
# weekend is flatter with a late-morning market bump.
WEEKDAY_HOURLY = np.array([
    0.10, 0.07, 0.05, 0.05, 0.10, 0.30,   # 00-05
    0.55, 0.85, 1.00, 0.80, 0.55, 0.50,   # 06-11
    0.55, 0.55, 0.55, 0.60, 0.75, 0.95,   # 12-17
    1.00, 0.80, 0.55, 0.35, 0.22, 0.14    # 18-23
])
WEEKEND_HOURLY = np.array([
    0.12, 0.08, 0.06, 0.05, 0.07, 0.15,
    0.25, 0.35, 0.50, 0.65, 0.75, 0.80,
    0.85, 0.80, 0.70, 0.65, 0.65, 0.70,
    0.65, 0.50, 0.35, 0.25, 0.18, 0.12
])

DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Vehicle-type mix (share of Total) - Nigeria skews heavier to
# keke/okada than the original Kaggle dataset's "Bike" category.
BASE_MIX = {"Car": 0.42, "KekeOkada": 0.33, "Bus": 0.18, "Truck": 0.07}

# Rainy season months in Lagos (heavier April-July, light Sept-Oct)
RAIN_MONTH_PROB = {1: 0.05, 2: 0.10, 3: 0.20, 4: 0.45, 5: 0.55, 6: 0.60,
                   7: 0.55, 8: 0.35, 9: 0.45, 10: 0.35, 11: 0.15, 12: 0.05}

# Random fuel-scarcity episodes: pick a few multi-day windows across the period
FUEL_SCARCITY_DAYS = set()
n_episodes = rng.integers(1, 3)
for _ in range(n_episodes):
    start_offset = rng.integers(0, NUM_DAYS - 7)
    length = rng.integers(3, 7)
    FUEL_SCARCITY_DAYS.update(range(start_offset, start_offset + length))


def congestion_bucket(ratio):
    """Map a load ratio (Total / capacity, adjusted) to a congestion label."""
    if ratio >= 0.90:
        return "1-Heavy"
    elif ratio >= 0.70:
        return "2-High"
    elif ratio >= 0.40:
        return "3-Normal"
    else:
        return "4-Low"


# ----------------------------------------------------------------------
# 2. GENERATE
# ----------------------------------------------------------------------

records = []
intervals_per_day = int(24 * 60 / INTERVAL_MINUTES)

for day_offset in range(NUM_DAYS):
    current_date = START_DATE + timedelta(days=day_offset)
    dow_index = current_date.weekday()          # 0=Mon ... 6=Sun
    day_name = DAY_NAMES[dow_index]
    is_weekend = dow_index >= 5
    is_market_day = dow_index in (1, 4)          # Tue/Fri as illustrative market days
    is_school_day = (not is_weekend)             # simple assumption; holidays not modeled

    # Daily rain draw
    rain_prob = RAIN_MONTH_PROB[current_date.month]
    is_rain_day = rng.random() < rain_prob
    rain_intensity = rng.uniform(0.3, 1.0) if is_rain_day else 0.0
    # Rain typically falls in a window (e.g. afternoon storms)
    rain_start_hour = rng.integers(13, 19) if is_rain_day else None
    rain_duration = rng.integers(1, 4) if is_rain_day else 0

    fuel_scarcity_today = day_offset in FUEL_SCARCITY_DAYS

    hourly_profile = WEEKEND_HOURLY if is_weekend else WEEKDAY_HOURLY

    for interval_idx in range(intervals_per_day):
        hour = interval_idx // 4
        minute = (interval_idx % 4) * 15
        time_str = f"{hour:02d}:{minute:02d}"
        base_hour_factor = hourly_profile[hour]

        raining_now = (
            is_rain_day and rain_start_hour is not None
            and rain_start_hour <= hour < rain_start_hour + rain_duration
        )

        for (seg_name, road_type, lanes, capacity, centrality,
             market_axis, school_axis) in SEGMENTS:

            # --- volume factor ---
            factor = base_hour_factor

            if market_axis and is_market_day:
                factor *= 1.15
            if school_axis and is_school_day and hour in (7, 8, 14, 15):
                factor *= 1.10
            if fuel_scarcity_today:
                # fewer cars move freely, but queues/okada substitution raises congestion
                factor *= 0.90
            if raining_now:
                factor *= 0.85  # fewer trips start, but existing trips slow down (handled below)

            # random noise per interval
            factor *= rng.normal(1.0, 0.08)
            factor = max(factor, 0.02)

            total = capacity * centrality * factor
            total = max(int(rng.normal(total, total * 0.07)), 0)

            # vehicle-type mix with small random perturbation
            mix = {k: max(v + rng.normal(0, 0.02), 0.02) for k, v in BASE_MIX.items()}
            if fuel_scarcity_today:
                mix["KekeOkada"] += 0.08
                mix["Car"] -= 0.06
            if raining_now:
                mix["Bus"] += 0.03
                mix["KekeOkada"] -= 0.03
            mix_sum = sum(mix.values())
            mix = {k: v / mix_sum for k, v in mix.items()}

            car_count = int(total * mix["Car"])
            keke_count = int(total * mix["KekeOkada"])
            bus_count = int(total * mix["Bus"])
            truck_count = int(total * mix["Truck"])
            total_count = car_count + keke_count + bus_count + truck_count

            # --- congestion ratio: volume load + rain penalty + fuel scarcity penalty ---
            load_ratio = total_count / capacity
            if raining_now:
                load_ratio *= 1.25   # rain slows flow -> effectively more congested for same volume
            if fuel_scarcity_today:
                load_ratio *= 1.10   # queues at fuel stations spill onto the road
            load_ratio = min(load_ratio, 1.5)

            situation = congestion_bucket(load_ratio)

            records.append({
                "Date": current_date.strftime("%Y-%m-%d"),
                "Time": time_str,
                "Day_of_week": day_name,
                "State": STATE,
                "Segment": seg_name,
                "Road_Type": road_type,
                "Lanes": lanes,
                "CarCount": car_count,
                "KekeOkadaCount": keke_count,
                "BusCount": bus_count,
                "TruckCount": truck_count,
                "Total": total_count,
                "Is_Market_Day": is_market_day,
                "Is_School_Hours": school_axis and is_school_day and hour in (7, 8, 14, 15),
                "Is_Raining": raining_now,
                "Rainfall_Intensity": round(rain_intensity if raining_now else 0.0, 2),
                "Is_Fuel_Scarcity": fuel_scarcity_today,
                "Traffic_Situation": situation,
            })

df = pd.DataFrame.from_records(records)

# ----------------------------------------------------------------------
# 3. SAVE
# ----------------------------------------------------------------------

output_path = "C:/Users/hp840 g3/Desktop/ML projects/3MTT CAPSTONE PROJECT/lagos_synthetic_traffic_dataset.csv"
df.to_csv(output_path, index=False)

print(f"Rows generated: {len(df):,}")
print(f"Segments: {df['Segment'].nunique()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print("\nTraffic_Situation distribution:")
print(df["Traffic_Situation"].value_counts(normalize=True).round(3))
print(f"\nSaved to: {output_path}")

Rows generated: 46,080
Segments: 8
Date range: 2025-01-01 to 2025-03-01

Traffic_Situation distribution:
Traffic_Situation
4-Low       0.489
3-Normal    0.373
2-High      0.102
1-Heavy     0.036
Name: proportion, dtype: float64

Saved to: C:/Users/hp840 g3/Desktop/ML projects/3MTT CAPSTONE PROJECT/lagos_synthetic_traffic_dataset.csv


In [11]:
df

,Date,Time,Day_of_week,State,Segment,Road_Type,Lanes,CarCount,KekeOkadaCount,BusCount,TruckCount,Total,Is_Market_Day,Is_School_Hours,Is_Raining,Rainfall_Intensity,Is_Fuel_Scarcity,Traffic_Situation
0,2025-01-01,00:00,Wednesday,Lagos,Third Mainland Bridge,highway,6,32,27,14,5,78,False,False,False,0.0,False,4-Low
1,2025-01-01,00:00,Wednesday,Lagos,Lekki-Epe Expressway,highway,6,27,21,12,5,65,False,False,False,0.0,False,4-Low
2,2025-01-01,00:00,Wednesday,Lagos,Ikorodu Road,arterial,4,22,19,9,3,53,False,False,False,0.0,False,4-Low
3,2025-01-01,00:00,Wednesday,Lagos,Agege Motor Road,arterial,3,15,12,6,3,36,False,False,False,0.0,False,4-Low
4,2025-01-01,00:00,Wednesday,Lagos,Apapa-Oshodi Expressway,highway,6,32,28,13,4,77,False,False,False,0.0,False,4-Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46075,2025-03-01,23:45,Saturday,Lagos,Agege Motor Road,arterial,3,19,14,7,2,42,False,False,False,0.0,False,4-Low
46076,2025-03-01,23:45,Saturday,Lagos,Apapa-Oshodi Expressway,highway,6,34,25,16,5,80,False,False,False,0.0,False,4-Low
46077,2025-03-01,23:45,Saturday,Lagos,Ozumba Mbadiwe,arterial,4,21,17,8,4,50,False,False,False,0.0,False,4-Low
46078,2025-03-01,23:45,Saturday,Lagos,Ajah-Sangotedo Road,arterial,3,13,9,4,1,27,False,False,False,0.0,False,4-Low
